# Q2.d) Logistic Regression for High Earner Prediction (12 points)

This notebook provides a complete analysis of high earner prediction using logistic regression.

## Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
import statsmodels.formula.api as smf

# Set random seed for reproducibility
np.random.seed(1818)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

In [ ]:
# Load the data
df = pd.read_csv('datasets/career_outcomes_survey.csv')

print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nFirst few rows:")
df.head()

## Part 1: Create Binary Target Variable

In [ ]:
# Calculate 75th percentile of current salary
salary_75th = df['salary_current'].quantile(0.75)
print(f"75th percentile of salary_current: ${salary_75th:,.2f}")

# Create binary target variable (recalculate to ensure consistency)
df['high_earner'] = (df['salary_current'] > salary_75th).astype(int)

# Calculate percentage of high earners
high_earner_pct = df['high_earner'].mean() * 100
print(f"\nPercentage of high earners: {high_earner_pct:.2f}%")

# Check class distribution
class_counts = df['high_earner'].value_counts().sort_index()
print("\nClass distribution:")
print(f"  Non-high earners (0): {class_counts[0]:,} ({class_counts[0]/len(df)*100:.1f}%)")
print(f"  High earners (1): {class_counts[1]:,} ({class_counts[1]/len(df)*100:.1f}%)")

# Assess imbalance
imbalance_ratio = class_counts[0] / class_counts[1]
print(f"\nImbalance ratio (majority/minority): {imbalance_ratio:.2f}:1")

if imbalance_ratio > 1.5:
    print("\n⚠️ Dataset is IMBALANCED (ratio > 1.5:1)")
    print("Consider: precision-recall curves, F1-score, and adjusting decision threshold")
else:
    print("\n✓ Dataset is relatively BALANCED")

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
class_counts.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'])
axes[0].set_title('High Earner Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('High Earner Status', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xticklabels(['Non-High Earner (0)', 'High Earner (1)'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Add count labels on bars
for i, v in enumerate(class_counts):
    axes[0].text(i, v + 20, f'{v:,}\n({v/len(df)*100:.1f}%)', 
                ha='center', va='bottom', fontweight='bold')

# Pie chart
colors = ['#3498db', '#e74c3c']
axes[1].pie(class_counts, labels=['Non-High Earner', 'High Earner'], 
           autopct='%1.1f%%', startangle=90, colors=colors,
           textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## Part 2: Build Logistic Regression Model

In [ ]:
# Create derived features to match the formula
# Create elite_university indicator (Top 10 or Top 100)
df['elite_university'] = df['university_tier'].isin(['Top 10', 'Top 100']).astype(int)

# Create stem_major indicator
stem_majors = ['Engineering', 'Computer Science', 'Data Science']
df['stem_major'] = df['major'].isin(stem_majors).astype(int)

# List all features that will be used in the model
features = [
    'years_experience',
    'gpa',
    'internship_count',
    'elite_university',
    'stem_major',
    'technical_skills',
    'leadership_roles',
    'major',
    'industry'
]

print("Features used in the model:")
print("\nContinuous/Binary features:")
for feat in ['years_experience', 'gpa', 'internship_count', 'elite_university', 
             'stem_major', 'technical_skills', 'leadership_roles']:
    print(f"  - {feat}")

print("\nCategorical features:")
print(f"  - C(major)")
print(f"  - C(industry)")

print(f"\nDataset shape: {df.shape}")
print(f"\nUnique majors: {df['major'].nunique()}")
print(f"Unique industries: {df['industry'].nunique()}")

In [ ]:
# Train-test split (70/30)
# Select relevant columns for the model
model_data = df[['high_earner', 'years_experience', 'gpa', 'internship_count', 
                 'elite_university', 'stem_major', 'technical_skills', 
                 'leadership_roles', 'major', 'industry']].copy()

# Handle missing values
print("Missing values check:")
print(model_data.isnull().sum())

if model_data.isnull().sum().sum() > 0:
    print("\n⚠️ Missing values detected. Filling numeric columns with median...")
    numeric_cols = ['years_experience', 'gpa', 'internship_count', 'technical_skills', 'leadership_roles']
    model_data[numeric_cols] = model_data[numeric_cols].fillna(model_data[numeric_cols].median())

# Split the data
train_df, test_df = train_test_split(
    model_data, test_size=0.30, random_state=1818, stratify=model_data['high_earner']
)

print(f"\nTraining set size: {len(train_df):,} ({len(train_df)/len(model_data)*100:.1f}%)")
print(f"Test set size: {len(test_df):,} ({len(test_df)/len(model_data)*100:.1f}%)")

# Check stratification
print("\nClass distribution in training set:")
print(train_df['high_earner'].value_counts(normalize=True).sort_index())
print("\nClass distribution in test set:")
print(test_df['high_earner'].value_counts(normalize=True).sort_index())

In [ ]:
# Note: Statsmodels doesn't require manual standardization for logistic regression
# The coefficients will be on the original scale, making interpretation easier
print("Using statsmodels - no standardization needed for interpretation")

In [ ]:
# Build logistic regression model using statsmodels
formula = 'high_earner ~ years_experience + gpa + internship_count + elite_university + stem_major + technical_skills + leadership_roles + C(major) + C(industry)'

print("Model Formula:")
print(formula)
print("\n" + "="*70)

# Fit the model on training data
log_reg = smf.logit(formula, data=train_df).fit()

print("\n✓ Logistic Regression model trained successfully!")
print(f"\nModel Summary:")
print(f"  - Number of observations: {log_reg.nobs:.0f}")
print(f"  - Number of parameters: {len(log_reg.params)}")
print(f"  - Log-Likelihood: {log_reg.llf:.2f}")
print(f"  - AIC: {log_reg.aic:.2f}")
print(f"  - BIC: {log_reg.bic:.2f}")
print(f"  - Pseudo R-squared (McFadden): {log_reg.prsquared:.4f}")

In [ ]:
# Display full model summary
print("\n" + "="*70)
print("FULL MODEL SUMMARY")
print("="*70)
print(log_reg.summary())

# Extract and display coefficients and odds ratios for main effects
print("\n" + "="*70)
print("COEFFICIENTS AND ODDS RATIOS (Main Effects)")
print("="*70)

# Create coefficients dataframe
coef_df = pd.DataFrame({
    'Coefficient': log_reg.params,
    'Std_Error': log_reg.bse,
    'z_value': log_reg.tvalues,
    'p_value': log_reg.pvalues,
    'Odds_Ratio': np.exp(log_reg.params)
})

# Sort by absolute coefficient value
coef_df['abs_coef'] = np.abs(coef_df['Coefficient'])
coef_df_sorted = coef_df.sort_values('abs_coef', ascending=False)

# Display main numerical predictors
main_predictors = ['years_experience', 'gpa', 'internship_count', 'elite_university', 
                   'stem_major', 'technical_skills', 'leadership_roles']
print("\nNumerical and Binary Predictors:")
for pred in main_predictors:
    if pred in coef_df.index:
        row = coef_df.loc[pred]
        sig = "***" if row['p_value'] < 0.001 else "**" if row['p_value'] < 0.01 else "*" if row['p_value'] < 0.05 else ""
        print(f"  {pred:25} Coef: {row['Coefficient']:8.4f}  OR: {row['Odds_Ratio']:7.4f}  p: {row['p_value']:.4f} {sig}")

print("\n" + "="*70)
print("\nInterpretation:")
print("- Positive coefficient → increases probability of being a high earner")
print("- Negative coefficient → decreases probability of being a high earner")
print("- Odds Ratio > 1 → positive association with high earner status")
print("- Odds Ratio < 1 → negative association with high earner status")
print("- Significance: *** p<0.001, ** p<0.01, * p<0.05")

In [ ]:
# Visualize coefficients for main effects only
main_predictors = ['years_experience', 'gpa', 'internship_count', 'elite_university', 
                   'stem_major', 'technical_skills', 'leadership_roles']

# Filter to main predictors
main_coefs = coef_df.loc[[p for p in main_predictors if p in coef_df.index]]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Coefficients bar plot
coef_sorted = main_coefs.sort_values('Coefficient')
colors = ['#e74c3c' if x < 0 else '#2ecc71' for x in coef_sorted['Coefficient']]
axes[0].barh(coef_sorted.index, coef_sorted['Coefficient'], color=colors, alpha=0.7)
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[0].set_xlabel('Coefficient Value', fontsize=12)
axes[0].set_title('Logistic Regression Coefficients (Main Effects)', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Add significance stars
for i, (idx, row) in enumerate(coef_sorted.iterrows()):
    sig = "***" if row['p_value'] < 0.001 else "**" if row['p_value'] < 0.01 else "*" if row['p_value'] < 0.05 else ""
    if sig:
        axes[0].text(row['Coefficient'], i, f" {sig}", va='center', fontsize=12, fontweight='bold')

# Odds ratios bar plot
or_sorted = main_coefs.sort_values('Odds_Ratio')
colors_or = ['#e74c3c' if x < 1 else '#2ecc71' for x in or_sorted['Odds_Ratio']]
axes[1].barh(or_sorted.index, or_sorted['Odds_Ratio'], color=colors_or, alpha=0.7)
axes[1].axvline(x=1, color='black', linestyle='--', linewidth=1, label='Odds Ratio = 1')
axes[1].set_xlabel('Odds Ratio', fontsize=12)
axes[1].set_title('Odds Ratios (exp(coefficient))', fontsize=14, fontweight='bold')
axes[1].legend(loc='best')
axes[1].grid(axis='x', alpha=0.3)

# Add significance stars
for i, (idx, row) in enumerate(or_sorted.iterrows()):
    sig = "***" if row['p_value'] < 0.001 else "**" if row['p_value'] < 0.01 else "*" if row['p_value'] < 0.05 else ""
    if sig:
        axes[1].text(row['Odds_Ratio'], i, f" {sig}", va='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## Part 3: Model Evaluation - Classification Metrics

In [ ]:
# Generate predictions using 0.5 threshold
y_prob = log_reg.predict(test_df)
y_pred_default = (y_prob >= 0.5).astype(int)

print(f"Predictions generated for {len(test_df):,} test samples")
print(f"\nPredicted class distribution (threshold=0.5):")
print(pd.Series(y_pred_default).value_counts().sort_index())

In [ ]:
# Calculate baseline accuracy (always predict majority class)
y_test = test_df['high_earner']
baseline_accuracy = y_test.value_counts().max() / len(y_test)

print(f"Baseline Accuracy (always predict majority class): {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"\nMajority class in test set: {y_test.value_counts().idxmax()}")
print(f"Majority class frequency: {y_test.value_counts().max()} / {len(y_test)}")

In [ ]:
# Calculate classification metrics
# Handle zero division warnings
zero_kw = {"zero_division": 0}

accuracy = accuracy_score(y_test, y_pred_default)
precision = precision_score(y_test, y_pred_default, **zero_kw)
recall = recall_score(y_test, y_pred_default, **zero_kw)
f1 = f1_score(y_test, y_pred_default, **zero_kw)

# Calculate baseline accuracy from training set
baseline_accuracy_train = max(train_df['high_earner'].mean(), 1 - train_df['high_earner'].mean())

# Create metrics_default dictionary
metrics_default = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Baseline_accuracy": baseline_accuracy_train,
}

print("\n" + "="*70)
print("CLASSIFICATION METRICS (Threshold = 0.5)")
print("="*70)
print(f"\nBaseline Accuracy (train):  {baseline_accuracy_train:.4f} ({baseline_accuracy_train*100:.2f}%)")
print(f"Model Accuracy:             {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision:                  {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:                     {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-Score:                   {f1:.4f}")
print("\n" + "="*70)

# Compare to baseline
improvement = (accuracy - baseline_accuracy_train) / baseline_accuracy_train * 100
print(f"\nModel improvement over baseline: {improvement:.2f}%")

# Display metrics_default dictionary
print("\nmetrics_default dictionary:")
for key, val in metrics_default.items():
    print(f"  {key}: {val:.4f}")

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, y_pred_default)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non-High Earner (0)', 'High Earner (1)'],
            yticklabels=['Non-High Earner (0)', 'High Earner (1)'],
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix (Threshold = 0.5)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Predicted Label', fontsize=14)
plt.ylabel('True Label', fontsize=14)

# Add percentage annotations
for i in range(2):
    for j in range(2):
        pct = cm[i, j] / cm.sum() * 100
        plt.text(j + 0.5, i + 0.7, f'({pct:.1f}%)', 
                ha='center', va='center', fontsize=11, color='gray')

plt.tight_layout()
plt.show()

# Print confusion matrix breakdown
tn, fp, fn, tp = cm.ravel()
print("\nConfusion Matrix Breakdown:")
print(f"  True Negatives (TN):  {tn:,}  - Correctly predicted non-high earners")
print(f"  False Positives (FP): {fp:,}  - Incorrectly predicted as high earners")
print(f"  False Negatives (FN): {fn:,}  - Missed high earners")
print(f"  True Positives (TP):  {tp:,}  - Correctly predicted high earners")

In [ ]:
# Detailed classification report
print("\nDetailed Classification Report:")
print("="*70)
print(classification_report(y_test, y_pred_default, 
                          target_names=['Non-High Earner (0)', 'High Earner (1)'],
                          digits=4))

## Part 4: ROC Analysis

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

print(f"ROC AUC Score: {roc_auc:.4f}")
print(f"\nNumber of threshold points: {len(thresholds_roc)}")

In [ ]:
# Plot ROC curve
fig1, ax = plt.subplots(figsize=(6,6))

# plot ROC curve
ax.plot(fpr, tpr, color="black", linewidth=2)
ax.plot([0,1], [0,1], linestyle="--", color="gray")  # diagonal baseline
ax.scatter(fpr, tpr, s=10, color="blue")  # optional: mark cutoff points

# label some cutoff points
for cutoff in [0.99, 0.9, 0.7, 0.5, 0.3, 0.1, 0.05, 0.075, 0.025, 0.01, 0.001, 0.003, 0.0]:
    # Find closest threshold value
    if cutoff == 0.0:
        idx = len(thresholds_roc) - 1  # Last threshold is usually 0.0
    else:
        idx = np.argmin(np.abs(thresholds_roc - cutoff))
    
    # Add text with slight offset and background for visibility
    ax.text(fpr[idx] + 0.02, tpr[idx] + 0.02, f"{cutoff:.3f}", 
            fontsize=9, color='red', weight='bold',
            bbox=dict(boxstyle="round,pad=0.2", facecolor='white', alpha=0.8))

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f'ROC Curve (AUC = {roc_auc:.4f})', fontsize=14, fontweight='bold')
plt.show(fig1)

print("\nROC Curve Interpretation:")
if roc_auc >= 0.9:
    print(f"  Excellent model (AUC ≥ 0.9)")
elif roc_auc >= 0.8:
    print(f"  Good model (0.8 ≤ AUC < 0.9)")
elif roc_auc >= 0.7:
    print(f"  Fair model (0.7 ≤ AUC < 0.8)")
else:
    print(f"  Poor model (AUC < 0.7)")

In [ ]:
# Find optimal threshold using Youden's J statistic
j_stats = tpr - fpr
optimal_idx = np.argmax(j_stats)
optimal_threshold = thresholds_roc[optimal_idx]
optimal_tpr = tpr[optimal_idx]
optimal_fpr = fpr[optimal_idx]

print("\n" + "="*70)
print("OPTIMAL THRESHOLD ANALYSIS (Youden's J Statistic)")
print("="*70)
print(f"\nOptimal Threshold: {optimal_threshold:.4f}")
print(f"J-Statistic at optimal: {j_stats[optimal_idx]:.4f}")
print(f"TPR (Recall) at optimal: {optimal_tpr:.4f}")
print(f"FPR at optimal: {optimal_fpr:.4f}")
print("\n" + "="*70)

# Generate predictions with optimal threshold
y_pred_optimal = (y_prob >= optimal_threshold).astype(int)

# Calculate metrics at optimal threshold
accuracy_opt = accuracy_score(y_test, y_pred_optimal)
precision_opt = precision_score(y_test, y_pred_optimal)
recall_opt = recall_score(y_test, y_pred_optimal)
f1_opt = f1_score(y_test, y_pred_optimal)

In [ ]:
# Plot ROC curve with optimal threshold point
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='#e74c3c', linewidth=2.5, 
         label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linewidth=1.5, 
         linestyle='--', label='Random Classifier')
plt.scatter(optimal_fpr, optimal_tpr, color='#2ecc71', s=200, 
           zorder=5, edgecolors='black', linewidths=2,
           label=f'Optimal Point (threshold={optimal_threshold:.3f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate (Recall)', fontsize=14)
plt.title('ROC Curve with Optimal Threshold', fontsize=16, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare metrics at default vs optimal threshold
comparison_df = pd.DataFrame({
    'Metric': ['Threshold', 'Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Default (0.5)': [0.5, accuracy, precision, recall, f1],
    'Optimal (Youden)': [optimal_threshold, accuracy_opt, precision_opt, recall_opt, f1_opt],
    'Change': [
        optimal_threshold - 0.5,
        accuracy_opt - accuracy,
        precision_opt - precision,
        recall_opt - recall,
        f1_opt - f1
    ]
})

print("\n" + "="*90)
print("METRICS COMPARISON: Default (0.5) vs Optimal Threshold")
print("="*90)
print(comparison_df.to_string(index=False))
print("\n" + "="*90)

# Highlight improvements
print("\nKey Changes:")
for idx, row in comparison_df.iterrows():
    if idx > 0:  # Skip threshold row
        change = row['Change']
        symbol = '↑' if change > 0 else '↓' if change < 0 else '→'
        print(f"  {row['Metric']:15} {symbol} {abs(change):.4f} ({abs(change)*100:.2f}%)")

In [ ]:
# Visualize metric comparison
fig, ax = plt.subplots(figsize=(12, 6))

metrics_to_plot = comparison_df[comparison_df['Metric'] != 'Threshold']
x = np.arange(len(metrics_to_plot))
width = 0.35

bars1 = ax.bar(x - width/2, metrics_to_plot['Default (0.5)'], width, 
              label='Default (0.5)', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, metrics_to_plot['Optimal (Youden)'], width,
              label=f'Optimal ({optimal_threshold:.3f})', color='#2ecc71', alpha=0.8)

ax.set_xlabel('Metrics', fontsize=14)
ax.set_ylabel('Score', fontsize=14)
ax.set_title('Metrics Comparison: Default vs Optimal Threshold', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot['Metric'], fontsize=12)
ax.legend(fontsize=12)
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
               f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Plot Precision-Recall Curve (important for imbalanced data)
precision_vals, recall_vals, thresholds_pr = precision_recall_curve(y_test, y_prob)
avg_precision = average_precision_score(y_test, y_prob)

plt.figure(figsize=(10, 8))
plt.plot(recall_vals, precision_vals, color='#9b59b6', linewidth=2.5,
         label=f'PR Curve (AP = {avg_precision:.4f})')
plt.axhline(y=y_test.mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'Baseline (No Skill) = {y_test.mean():.3f}')
plt.xlabel('Recall (Sensitivity)', fontsize=14)
plt.ylabel('Precision', fontsize=14)
plt.title('Precision-Recall Curve', fontsize=16, fontweight='bold')
plt.legend(loc='best', fontsize=12)
plt.grid(alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.tight_layout()
plt.show()

print(f"\nAverage Precision Score: {avg_precision:.4f}")
print(f"Baseline (random classifier): {y_test.mean():.4f}")
print("\nNote: For imbalanced datasets, PR curves are more informative than ROC curves.")

## Part 5: Probability Calibration

In [ ]:
# Clip probabilities to ensure they're in [0, 1] range
y_prob_clipped = np.clip(y_prob, 0, 1)

# Create calibration plot
prob_true, prob_pred = calibration_curve(y_test, y_prob_clipped, n_bins=10, strategy='quantile')

plt.figure(figsize=(10, 8))
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, markersize=10,
         color='#e74c3c', label='Logistic Regression')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=2,
         label='Perfectly Calibrated')
plt.xlabel('Mean Predicted Probability', fontsize=14)
plt.ylabel('Fraction of Positives (True Probability)', fontsize=14)
plt.title('Calibration Plot (Reliability Diagram)', fontsize=16, fontweight='bold')
plt.legend(loc='upper left', fontsize=12)
plt.grid(alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.tight_layout()
plt.show()

# Calculate calibration metrics
calibration_error = np.mean(np.abs(prob_pred - prob_true))
print(f"\nMean Calibration Error: {calibration_error:.4f}")

if calibration_error < 0.05:
    print("✓ Probabilities are WELL-CALIBRATED (error < 0.05)")
elif calibration_error < 0.10:
    print("⚠️ Probabilities are MODERATELY CALIBRATED (0.05 ≤ error < 0.10)")
else:
    print("✗ Probabilities are POORLY CALIBRATED (error ≥ 0.10)")

In [ ]:
# Show distribution of predicted probabilities for both classes
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Overall histogram
axes[0, 0].hist(y_prob, bins=30, edgecolor='black', alpha=0.7, color='#3498db')
axes[0, 0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Default Threshold (0.5)')
axes[0, 0].axvline(x=optimal_threshold, color='green', linestyle='--', linewidth=2, 
                   label=f'Optimal Threshold ({optimal_threshold:.3f})')
axes[0, 0].set_xlabel('Predicted Probability', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('Overall Distribution of Predicted Probabilities', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(alpha=0.3)

# Separate distributions by actual class
prob_class_0 = y_prob[y_test == 0]
prob_class_1 = y_prob[y_test == 1]

axes[0, 1].hist(prob_class_0, bins=30, edgecolor='black', alpha=0.7, color='#3498db', label='Actual: Non-High Earner')
axes[0, 1].set_xlabel('Predicted Probability', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)
axes[0, 1].set_title('Distribution for Non-High Earners (Class 0)', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(alpha=0.3)

axes[1, 0].hist(prob_class_1, bins=30, edgecolor='black', alpha=0.7, color='#e74c3c', label='Actual: High Earner')
axes[1, 0].set_xlabel('Predicted Probability', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)
axes[1, 0].set_title('Distribution for High Earners (Class 1)', fontsize=14, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# Overlapping distributions
axes[1, 1].hist(prob_class_0, bins=30, alpha=0.5, color='#3498db', label='Non-High Earner (0)', density=True)
axes[1, 1].hist(prob_class_1, bins=30, alpha=0.5, color='#e74c3c', label='High Earner (1)', density=True)
axes[1, 1].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold (0.5)')
axes[1, 1].set_xlabel('Predicted Probability', fontsize=12)
axes[1, 1].set_ylabel('Density', fontsize=12)
axes[1, 1].set_title('Overlapping Distributions (Normalized)', fontsize=14, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print("\nProbability Distribution Statistics:")
print("="*70)
print(f"\nNon-High Earners (Class 0):")
print(f"  Mean probability: {prob_class_0.mean():.4f}")
print(f"  Std deviation: {prob_class_0.std():.4f}")
print(f"  Median: {np.median(prob_class_0):.4f}")
print(f"  Range: [{prob_class_0.min():.4f}, {prob_class_0.max():.4f}]")

print(f"\nHigh Earners (Class 1):")
print(f"  Mean probability: {prob_class_1.mean():.4f}")
print(f"  Std deviation: {prob_class_1.std():.4f}")
print(f"  Median: {np.median(prob_class_1):.4f}")
print(f"  Range: [{prob_class_1.min():.4f}, {prob_class_1.max():.4f}]")

# Check separation
separation = prob_class_1.mean() - prob_class_0.mean()
print(f"\nMean probability separation: {separation:.4f}")
if separation > 0.3:
    print("✓ Good separation between classes")
else:
    print("⚠️ Limited separation between classes")

## Model Results Storage

# Calculate Brier score
brier = brier_score_loss(y_test, y_prob)

print("=" * 70)
print("MODEL RESULTS SUMMARY")
print("=" * 70)

# Create q2_models dictionary to persist all results
q2_models = {}

# Store the model
q2_models["logit_model"] = log_reg

# Store all metrics
q2_models["logit_metrics"] = {
    "default": metrics_default,
    "brier_score": brier,
    "roc_curve": (fpr, tpr, thresholds_roc),
    "calibration": {"prob_true": prob_true, "prob_pred": prob_pred},
}

print("\n1. STORED MODEL:")
print(f"   - Model type: {type(log_reg).__name__}")
print(f"   - Formula: {formula}")

print("\n2. STORED METRICS (default threshold = 0.5):")
for key, val in metrics_default.items():
    print(f"   - {key}: {val:.4f}")

print(f"\n3. BRIER SCORE:")
print(f"   - Brier Score: {brier:.4f}")
print(f"   - Lower is better (perfect = 0.0, random = 0.25)")

print(f"\n4. ROC CURVE DATA:")
print(f"   - FPR points: {len(fpr)}")
print(f"   - TPR points: {len(tpr)}")
print(f"   - Thresholds: {len(thresholds_roc)}")
print(f"   - ROC AUC: {roc_auc:.4f}")

print(f"\n5. CALIBRATION DATA:")
print(f"   - Calibration bins: {len(prob_true)}")
print(f"   - Mean calibration error: {calibration_error:.4f}")

print("\n" + "=" * 70)
print("\n✓ All model results stored in 'q2_models' dictionary!")
print("\nAccess components:")
print("  - q2_models['logit_model'] - the fitted model")
print("  - q2_models['logit_metrics']['default'] - default metrics")
print("  - q2_models['logit_metrics']['brier_score'] - Brier score")
print("  - q2_models['logit_metrics']['roc_curve'] - ROC curve data")
print("  - q2_models['logit_metrics']['calibration'] - calibration data")

In [ ]:
# Final comprehensive summary
print("\n" + "="*90)
print("Q2.d) HIGH EARNER PREDICTION - COMPREHENSIVE SUMMARY")
print("="*90)

print("\n1. TARGET VARIABLE & CLASS BALANCE:")
print(f"   - High earner threshold: ${salary_75th:,.2f} (75th percentile)")
print(f"   - High earner percentage: {high_earner_pct:.2f}%")
print(f"   - Class imbalance ratio: {imbalance_ratio:.2f}:1")
print(f"   - Dataset status: {'IMBALANCED' if imbalance_ratio > 1.5 else 'BALANCED'}")

print("\n2. MODEL CONFIGURATION:")
print(f"   - Algorithm: Logistic Regression (statsmodels)")
print(f"   - Formula: high_earner ~ years_experience + gpa + internship_count +")
print(f"              elite_university + stem_major + technical_skills +")
print(f"              leadership_roles + C(major) + C(industry)")
print(f"   - Number of parameters: {len(log_reg.params)}")
print(f"   - Train/Test split: 70/30 (stratified)")
print(f"   - Pseudo R-squared: {log_reg.prsquared:.4f}")

print("\n3. TOP PREDICTIVE FEATURES:")
main_predictors = ['years_experience', 'gpa', 'internship_count', 'elite_university', 
                   'stem_major', 'technical_skills', 'leadership_roles']
main_coefs = coef_df.loc[[p for p in main_predictors if p in coef_df.index]]
top_3 = main_coefs.nlargest(3, 'abs_coef')
for i, (idx, row) in enumerate(top_3.iterrows(), 1):
    sig = "***" if row['p_value'] < 0.001 else "**" if row['p_value'] < 0.01 else "*" if row['p_value'] < 0.05 else ""
    print(f"   {i}. {idx:25} (coef={row['Coefficient']:7.4f}, OR={row['Odds_Ratio']:.4f}) {sig}")

print("\n4. PERFORMANCE METRICS:")
print(f"   Baseline Accuracy:        {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"   Model Accuracy (t=0.5):   {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Model Accuracy (optimal): {accuracy_opt:.4f} ({accuracy_opt*100:.2f}%)")
improvement = (accuracy - baseline_accuracy) / baseline_accuracy * 100
print(f"   Improvement over baseline: {improvement:.2f}%")

print("\n5. ROC & THRESHOLD ANALYSIS:")
print(f"   ROC AUC Score:      {roc_auc:.4f}")
print(f"   Optimal Threshold:  {optimal_threshold:.4f}")
print(f"   Default Threshold:  0.5000")
print(f"   Threshold shift:    {optimal_threshold - 0.5:+.4f}")

print("\n6. CLASSIFICATION PERFORMANCE (at optimal threshold):")
print(f"   Precision: {precision_opt:.4f}  (What % of predicted high earners are correct?)")
print(f"   Recall:    {recall_opt:.4f}  (What % of actual high earners did we catch?)")
print(f"   F1-Score:  {f1_opt:.4f}  (Harmonic mean of precision and recall)")

print("\n7. PROBABILITY CALIBRATION:")
print(f"   Mean Calibration Error: {calibration_error:.4f}")
print(f"   Calibration Status: {'WELL-CALIBRATED' if calibration_error < 0.05 else 'NEEDS IMPROVEMENT'}")
print(f"   Avg Precision Score: {avg_precision:.4f}")

print("\n8. MODEL FIT STATISTICS:")
print(f"   Log-Likelihood: {log_reg.llf:.2f}")
print(f"   AIC: {log_reg.aic:.2f}")
print(f"   BIC: {log_reg.bic:.2f}")

print("\n9. KEY INSIGHTS:")
if roc_auc >= 0.8:
    print("   ✓ Model shows strong discriminative ability (AUC ≥ 0.8)")
else:
    print("   ⚠️ Model has moderate discriminative ability")

if accuracy > baseline_accuracy:
    print(f"   ✓ Model significantly outperforms baseline by {improvement:.1f}%")
    
if optimal_threshold != 0.5:
    print(f"   ⚠️ Consider using threshold={optimal_threshold:.3f} instead of default 0.5")

if imbalance_ratio > 1.5:
    print("   ⚠️ Dataset imbalance: PR curve and F1-score are more reliable metrics")

print("\n" + "="*90)
print("\n✓ Analysis Complete!")